# Creating the Test Data

In [1]:
import numpy as np

# Suppress scientific notation for cleaner output
np.set_printoptions(suppress=True)

# Generate 20 random floating-point numbers between -50 and 150
# These represent neural network parameters (weights/activations) we want to quantize
params = np.random.uniform(low=-50, high=150, size=20)

# Set up test cases at the beginning of the array for easier debugging:
# params[0] = maximum value (will test upper bound of quantization)
# params[1] = minimum value (will test lower bound of quantization)
# params[2] = zero (will test zero-point handling)
params[0] = params.max() + 1
params[1] = params.min() - 1
params[2] = 0

# Round to 2 decimal places for readability
params = np.round(params, 2)

print(params)

[144.35 -41.42   0.   -38.23   7.57 139.5    4.39  72.44  72.52  87.6
   7.13 100.13  -1.52  17.98  -7.99  82.88 -40.42  62.44 143.35  59.08]


We want to compress these to 8-bit integers (0-255)
to save memory while preserving accuracy

# Quantization Helper Functions and Main Algorithms

## The Clamp Function (Boundary Enforcement)

In [2]:
def clamp(params_q: np.array, lower_bound: int, upper_bound: int) -> np.array:
    """
    Clamp (limit) values to stay within valid quantization range.
    
    Visual representation:
    If we're using 8-bit unsigned integers (0-255):
    
        -10  →  0   (too low, clamp to 0)
         150 →  150 (within range, keep as is)
         300 →  255 (too high, clamp to 255)
    
    This prevents overflow/underflow during quantization.
    """
    params_q[params_q < lower_bound] = lower_bound  # Set all values below min to min
    params_q[params_q > upper_bound] = upper_bound  # Set all values above max to max
    return params_q

## Asymmetric Quantization (Uses Full Range)

In [3]:
def asymmetric_quantization(params: np.array, bits: int) -> tuple[np.array, float, int]:
    """
    Asymmetric quantization maps the full range of input values to the full quantization range.
    
    KEY IDEA: Use a scale factor AND a zero-point offset
    
    Visual Example with 8-bit (0-255):
    
    Original range:     [-40.1  ←--------→  127.48]
                            ↓                   ↓
    Quantized range:    [  0   ←--------→   255  ]
    
    Formula: quantized = round(original / scale + zero_point)
    
    Where:
    - scale = (max - min) / (2^bits - 1)  # How much each quantized step represents
    - zero_point = where the original zero maps to in quantized space
    """
    
    # Step 1: Find the range of our data
    alpha = np.max(params)  # Maximum value (e.g., 127.48)
    beta = np.min(params)   # Minimum value (e.g., -40.1)
    
    # Step 2: Calculate scale factor
    # For 8-bit: (127.48 - (-40.1)) / 255 = 0.657
    # This means each quantized integer step represents 0.657 in original scale
    scale = (alpha - beta) / (2**bits - 1)
    
    # Step 3: Calculate zero-point
    # This determines where "0" in the original data maps to in quantized space
    # zero = -round(min / scale) ensures min value maps to 0
    zero = -1 * np.round(beta / scale)
    
    # Step 4: Define valid range for quantized values
    lower_bound, upper_bound = 0, 2**bits - 1  # For 8-bit: [0, 255]
    
    # Step 5: Quantize!
    # Formula: q = clamp(round(original / scale + zero_point))
    quantized = clamp(
        np.round(params / scale + zero),  # Scale and shift
        lower_bound, 
        upper_bound
    ).astype(np.int32)
    
    return quantized, scale, zero

    Original value: 89.74
       ↓ (divide by scale 0.657)
       136.6
       ↓ (add zero-point 61)
       197.6
       ↓ (round)
       198  ← Final quantized value

## Symmetric Quantization (Centered at Zero)

In [4]:
def symmetric_quantization(params: np.array, bits: int) -> tuple[np.array, float]:
    """
    Symmetric quantization centers the quantization range around zero.
    
    KEY IDEA: Use only a scale factor (NO zero-point offset)
    The quantization range is symmetric: [-127, 127] for 8-bit signed
    
    Visual Example with 8-bit signed (-128 to 127, but we use -127 to 127 for symmetry):
    
    Original range:     [-127.48  ←---0---→  127.48]
                             ↓      ↓          ↓
    Quantized range:    [ -127   ←-0-→   127     ]
    
    Formula: quantized = round(original / scale)
    
    Where:
    - scale = max(|values|) / (2^(bits-1) - 1)  # Scale based on largest absolute value
    - Zero in original space maps exactly to 0 in quantized space
    
    ADVANTAGE: Better for data centered around zero (common in neural networks)
    DISADVANTAGE: May waste range if data is not centered (e.g., all positive)
    """
    
    # Step 1: Find the maximum absolute value
    # This ensures our scale can represent both the largest positive AND negative value
    alpha = np.max(np.abs(params))  # e.g., max(|127.48|, |-40.1|) = 127.48
    
    # Step 2: Calculate scale factor
    # For 8-bit: 127.48 / 127 = 1.004
    # This means each quantized integer step represents 1.004 in original scale
    scale = alpha / (2**(bits-1) - 1)
    
    # Step 3: Define valid range for quantized values
    # For 8-bit signed: [-128, 127], but we use [-127, 127] for perfect symmetry
    lower_bound = -2**(bits-1)      # -128 for 8-bit
    upper_bound = 2**(bits-1) - 1   # 127 for 8-bit
    
    # Step 4: Quantize!
    # Formula: q = clamp(round(original / scale))
    # Note: No zero-point! Zero maps to zero naturally
    quantized = clamp(
        np.round(params / scale),  # Just scale, no offset
        lower_bound, 
        upper_bound
    ).astype(np.int32)
    
    return quantized, scale

    ASYMMETRIC (uses full 0-255 range):
    Original: [-40.1 ═══════════ 0 ════════════════ 127.48]
                  ↓              ↓                     ↓
    Quantized: [  0  ═══════════ 61 ═══════════════  255  ]
    Uses: scale + zero-point
    
    SYMMETRIC (centered at zero, uses -127 to 127):
    Original: [-127.48 ════════ 0 ════════ 127.48]
                   ↓            ↓            ↓
    Quantized: [ -127  ════════ 0 ════════  127  ]
    Uses: scale only

## Dequantization Functions (Converting Back)

In [5]:
def asymmetric_dequantize(params_q: np.array, scale: float, zero: int) -> np.array:
    """
    Convert asymmetric quantized values back to floating point.
    
    Formula: original ≈ (quantized - zero_point) * scale
    
    Example:
    quantized = 198, scale = 0.657, zero = 61
    original ≈ (198 - 61) * 0.657 = 137 * 0.657 ≈ 90.03
    (actual original was 89.74, so small error introduced)
    """
    return (params_q - zero) * scale


def symmetric_dequantize(params_q: np.array, scale: float) -> np.array:
    """
    Convert symmetric quantized values back to floating point.
    
    Formula: original ≈ quantized * scale
    
    Example:
    quantized = 89, scale = 1.004
    original ≈ 89 * 1.004 ≈ 89.36
    (actual original was 89.74, so small error introduced)
    """
    return params_q * scale

## Error Calculation

In [6]:
def quantization_error(params: np.array, params_q: np.array):
    """
    Calculate Mean Squared Error (MSE) between original and dequantized values.
    
    MSE = average of (original - reconstructed)²
    
    Lower MSE = better quantization (less information loss)
    """
    return np.mean((params - params_q)**2)

# Running the Quantization and Examining Results

### QUANTIZE THE PARAMETERS

In [8]:
# Perform ASYMMETRIC quantization to 8 bits (0-255 range)
(asymmetric_q, asymmetric_scale, asymmetric_zero) = asymmetric_quantization(params, 8)

# Perform SYMMETRIC quantization to 8 bits (-127 to 127 range)
(symmetric_q, symmetric_scale) = symmetric_quantization(params, 8)

### DISPLAY THE RESULTS

In [11]:
print(f'Original:')
print(np.round(params, 2))
print()

print(f'Asymmetric scale: {asymmetric_scale}, zero: {asymmetric_zero}')
print(asymmetric_q)
print()

print(f'Symmetric scale: {symmetric_scale}')
print(symmetric_q)

Original:
[144.35 -41.42   0.   -38.23   7.57 139.5    4.39  72.44  72.52  87.6
   7.13 100.13  -1.52  17.98  -7.99  82.88 -40.42  62.44 143.35  59.08]

Asymmetric scale: 0.7285098039215686, zero: 57.0
[255   0  57   5  67 248  63 156 157 177  67 194  55  82  46 171   2 143
 254 138]

Symmetric scale: 1.1366141732283463
[127 -36   0 -34   7 123   4  64  64  77   6  88  -1  16  -7  73 -36  55
 126  52]


    ORIGINAL VALUES (32-bit floats):
    [127.48, -40.1, 0.0, 89.74, 124.38, -39.1, 126.48, 21.2, ...]
       ↓                          ↓                          ↓
       
    ASYMMETRIC QUANTIZATION (8-bit unsigned: 0-255):
       Parameters: scale = 0.657, zero_point = 61
       
       Process for each value:
       127.48 → 127.48/0.657 + 61 → 194 + 61 → 255 ✓
       -40.1  → -40.1/0.657 + 61  → -61 + 61  → 0   ✓
       0.0    → 0/0.657 + 61      → 0 + 61    → 61  ✓
       89.74  → 89.74/0.657 + 61  → 137 + 61  → 198 ✓
       
       Result: [255, 0, 61, 198, 250, 2, 253, 93, 6, 250, ...]
       
       ↓                          ↓                          ↓
       
    SYMMETRIC QUANTIZATION (8-bit signed: -127 to 127):
       Parameters: scale = 1.004
       
       Process for each value:
       127.48 → 127.48/1.004 → 127  ✓
       -40.1  → -40.1/1.004  → -40  ✓
       0.0    → 0/1.004      → 0    ✓
       89.74  → 89.74/1.004  → 89   ✓
       
       Result: [127, -40, 0, 89, 124, -39, 126, 21, -36, 124, ...]

#  Dequantization and Error Analysis

### DEQUANTIZE: Convert quantized integers back to floating point

In [12]:
# Convert asymmetric quantized values back using: (q - zero) * scale
params_deq_asymmetric = asymmetric_dequantize(asymmetric_q, asymmetric_scale, asymmetric_zero)

# Convert symmetric quantized values back using: q * scale
params_deq_symmetric = symmetric_dequantize(symmetric_q, symmetric_scale)

### COMPARE: Original vs Reconstructed values

In [13]:
print(f'Original:')
print(np.round(params, 2))
print('')

print(f'Dequantize Asymmetric:')
print(np.round(params_deq_asymmetric, 2))
print('')

print(f'Dequantize Symmetric:')
print(np.round(params_deq_symmetric, 2))

Original:
[144.35 -41.42   0.   -38.23   7.57 139.5    4.39  72.44  72.52  87.6
   7.13 100.13  -1.52  17.98  -7.99  82.88 -40.42  62.44 143.35  59.08]

Dequantize Asymmetric:
[144.24 -41.53   0.   -37.88   7.29 139.15   4.37  72.12  72.85  87.42
   7.29  99.81  -1.46  18.21  -8.01  83.05 -40.07  62.65 143.52  59.01]

Dequantize Symmetric:
[144.35 -40.92   0.   -38.64   7.96 139.8    4.55  72.74  72.74  87.52
   6.82 100.02  -1.14  18.19  -7.96  82.97 -40.92  62.51 143.21  59.1 ]


    VALUE #1: 89.74
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    ASYMMETRIC PATH:
    Original:    89.74 (32-bit float)
        ↓ quantize
    Quantized:   198 (8-bit unsigned int) 
        ↓ dequantize: (198 - 61) * 0.657
    Reconstructed: 90.03
    ERROR: |89.74 - 90.03| = 0.29 ✓ pretty close!
    
    SYMMETRIC PATH:
    Original:    89.74 (32-bit float)
        ↓ quantize
    Quantized:   89 (8-bit signed int)
        ↓ dequantize: 89 * 1.004
    Reconstructed: 89.36
    ERROR: |89.74 - 89.36| = 0.38 ✓ also close!
    
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    VALUE #2: -40.1
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    ASYMMETRIC PATH:
    Original:    -40.1 (32-bit float)
        ↓ quantize
    Quantized:   0 (8-bit unsigned int)
        ↓ dequantize: (0 - 61) * 0.657
    Reconstructed: -40.09
    ERROR: |-40.1 - (-40.09)| = 0.01 ✓ excellent!
    
    SYMMETRIC PATH:
    Original:    -40.1 (32-bit float)
        ↓ quantize
    Quantized:   -40 (8-bit signed int)
        ↓ dequantize: -40 * 1.004
    Reconstructed: -40.15
    ERROR: |-40.1 - (-40.15)| = 0.05 ✓ very good!
    ```
    
    Full Array Comparison:
    
    Index | Original | Asymmetric | Symmetric | Asym Error | Sym Error
    ------|----------|------------|-----------|------------|----------
      0   | 127.48   | 127.49     | 127.48    |   0.01     |   0.00   
      1   | -40.10   | -40.09     | -40.15    |   0.01     |   0.05
      2   |   0.00   |   0.00     |   0.00    |   0.00     |   0.00   
      3   |  89.74   |  90.03     |  89.34    |   0.29     |   0.40
      4   | 124.38   | 124.21     | 124.47    |   0.17     |   0.09


# Quantization Error Measurement (Final Analysis)

In [15]:
# Calculate MEAN SQUARED ERROR (MSE)
# MSE measures the average squared difference between original and reconstructed values
# Lower MSE = better quantization quality

print(f'{"Asymmetric error: ":>20}{np.round(quantization_error(params, params_deq_asymmetric), 2)}')
print(f'{"Symmetric error: ":>20}{np.round(quantization_error(params, params_deq_symmetric), 2)}')

  Asymmetric error: 0.05
   Symmetric error: 0.07


    Why is asymmetric better here?
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    Our data range: [-40.1 to 127.48]
                      ↑              ↑
                 uneven distribution!
         (more positive values than negative)
    
    ASYMMETRIC: 
    - Uses full 0-255 range efficiently
    - Maps [-40.1, 127.48] → [0, 255]
    - NO wasted quantization levels ✓
    
    SYMMETRIC:
    - Reserves equal space for positive/negative
    - Maps [-127.48, 127.48] → [-127, 127]
    - Wastes range from -127 to -40 (unused!) 
      
      Visual:
      [-127 ←--WASTED--→ -40 ←--USED--→ 0 ←--USED--→ 127]
    
    
    When to Use Each Method:
    
    USE ASYMMETRIC when:
    ✓ Data has uneven positive/negative distribution
    ✓ Data is all positive (e.g., 0-255 image pixels)
    ✓ Data is all negative
    ✓ You need maximum precision for the actual data range
    
    Examples:
    - Image pixel values: [0, 255]
    - ReLU activation outputs: [0, ∞)
    - Probabilities: [0, 1]
    
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    USE SYMMETRIC when:
    ✓ Data is centered around zero
    ✓ Data has roughly equal positive/negative values
    ✓ You want zero to map exactly to zero (important for some operations)
    ✓ Simpler hardware implementation (no zero-point tracking)
    
    Examples:
    - Neural network weights after normalization: [-σ, +σ]
    - Tanh activation outputs: [-1, 1]
    - Standardized data: mean=0, std=1

    Summary:
    
    QUANTIZATION TRADE-OFFS:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    BENEFITS:
    ✓ 75% memory reduction (32-bit → 8-bit)
    ✓ Faster computation (integer math vs float)
    ✓ Lower power consumption
    ✓ Enables running large models on mobile/edge devices
    
    COSTS:
    ✗ Precision loss (MSE ≈ 0.03-0.08 in our example)
    ✗ Additional overhead (storing scale, zero-point)
    ✗ Requires careful tuning for each use case
    
    REAL-WORLD IMPACT:
    In neural networks, quantization can:
    - Reduce model size from 500MB → 125MB
    - Speed up inference by 2-4x
    - With only 1-2% accuracy drop!